In [14]:
# 1. IMPORT LIBRARIES AND CONFIGURE THE MEDALLION PIPELINE
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Use UTC consistently for timestamps
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Project configuration
BRONZE_PATH = "Files/bronze/synthea"
SILVER_SCHEMA = "silver"
SOURCE_SYSTEM = "Synthea CSV November 2021"

# Create the Silver schema in the Lakehouse
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

print("Configuration completed.")
print(f"Bronze source: {BRONZE_PATH}")
print(f"Silver schema: {SILVER_SCHEMA}")

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 16, Finished, Available, Finished, False)

Configuration completed.
Bronze source: Files/bronze/synthea
Silver schema: silver


In [15]:
# 2. INGEST THE RAW BRONZE PATIENTS CSV FILE

SOURCE_FILE = "patients.csv"
SOURCE_TABLE = "patients"

patient_file_path = (
    f"{BRONZE_PATH}/{SOURCE_FILE}"
)

patients_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(patient_file_path)
)

print(
    f"Source table: {SOURCE_TABLE}"
)

print(
    f"Source file: {patient_file_path}"
)

print(
    f"Rows loaded: {patients_raw.count():,}"
)

print(
    f"Columns loaded: {len(patients_raw.columns)}"
)

patients_raw.printSchema()

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 17, Finished, Available, Finished, False)

Source table: patients
Source file: Files/bronze/synthea/patients.csv
Rows loaded: 1,163
Columns loaded: 25
root
 |-- Id: string (nullable = true)
 |-- BIRTHDATE: string (nullable = true)
 |-- DEATHDATE: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- DRIVERS: string (nullable = true)
 |-- PASSPORT: string (nullable = true)
 |-- PREFIX: string (nullable = true)
 |-- FIRST: string (nullable = true)
 |-- LAST: string (nullable = true)
 |-- SUFFIX: string (nullable = true)
 |-- MAIDEN: string (nullable = true)
 |-- MARITAL: string (nullable = true)
 |-- RACE: string (nullable = true)
 |-- ETHNICITY: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- BIRTHPLACE: string (nullable = true)
 |-- ADDRESS: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- LAT: string (nullable = true)
 |-- LON: string (nullable = true)
 |-- HEALTHCARE_EXP

In [16]:
# 3. STANDARDIZE PATIENT COLUMN NAMES

import re

# Define important patient key names
key_column_names = {
    "Id": "id",
    "ID": "id"
}


def standardize_column_name(column_name):

    if column_name in key_column_names:
        return key_column_names[column_name]

    # Convert spaces and symbols to underscores
    # and standardize names to lowercase
    cleaned_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name
    )

    return (
        cleaned_name
        .strip("_")
        .lower()
    )


patients_standardized = patients_raw

for old_name in patients_raw.columns:

    new_name = standardize_column_name(
        old_name
    )

    patients_standardized = (
        patients_standardized
        .withColumnRenamed(
            old_name,
            new_name
        )
    )


# Stop if column-name standardization
# accidentally creates duplicate columns
if (
    len(patients_standardized.columns)
    !=
    len(set(patients_standardized.columns))
):

    raise ValueError(
        "Duplicate column names were created "
        "while standardizing the patients table."
    )


print(
    f"Patients: "
    f"{len(patients_raw.columns)} source columns -> "
    f"{len(patients_standardized.columns)} "
    f"standardized columns"
)

print(
    patients_standardized.columns
)

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 18, Finished, Available, Finished, False)

Patients: 25 source columns -> 25 standardized columns
['id', 'birthdate', 'deathdate', 'ssn', 'drivers', 'passport', 'prefix', 'first', 'last', 'suffix', 'maiden', 'marital', 'race', 'ethnicity', 'gender', 'birthplace', 'address', 'city', 'state', 'county', 'zip', 'lat', 'lon', 'healthcare_expenses', 'healthcare_coverage']


In [17]:
# 4. CLEAN PATIENT STRING VALUES

def clean_string_values(df):

    cleaned_df = df

    for column_name in df.columns:

        cleaned_df = (
            cleaned_df
            .withColumn(
                column_name,
                F.when(
                    F.trim(
                        F.col(column_name)
                    ) == "",
                    F.lit(None)
                )
                .otherwise(
                    F.trim(
                        F.col(column_name)
                    )
                )
            )
        )

    return cleaned_df


patient_count_before_cleaning = (
    patients_standardized.count()
)

patients_cleaned = clean_string_values(
    patients_standardized
)

patient_count_after_cleaning = (
    patients_cleaned.count()
)


print(
    f"Patients: "
    f"{patient_count_before_cleaning:,} rows "
    f"before cleaning, "
    f"{patient_count_after_cleaning:,} rows "
    f"after cleaning"
)

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 19, Finished, Available, Finished, False)

Patients: 1,163 rows before cleaning, 1,163 rows after cleaning


In [18]:
# 5. TRANSFORM PATIENT DATA TYPES

patients_silver = (
    patients_cleaned

    # Convert birth and death dates
    # from text to date values
    .withColumn(
        "birthdate",
        F.to_date(
            F.col("birthdate"),
            "yyyy-MM-dd"
        )
    )
    .withColumn(
        "deathdate",
        F.to_date(
            F.col("deathdate"),
            "yyyy-MM-dd"
        )
    )

    # Convert geographic coordinates
    # to numeric values
    .withColumn(
        "lat",
        F.col("lat").cast(
            T.DoubleType()
        )
    )
    .withColumn(
        "lon",
        F.col("lon").cast(
            T.DoubleType()
        )
    )

    # Convert financial values to
    # fixed-precision decimals
    .withColumn(
        "healthcare_expenses",
        F.col(
            "healthcare_expenses"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
    .withColumn(
        "healthcare_coverage",
        F.col(
            "healthcare_coverage"
        ).cast(
            T.DecimalType(18, 2)
        )
    )
)

patients_silver.printSchema()

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 20, Finished, Available, Finished, False)

root
 |-- id: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- deathdate: date (nullable = true)
 |-- ssn: string (nullable = true)
 |-- drivers: string (nullable = true)
 |-- passport: string (nullable = true)
 |-- prefix: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- suffix: string (nullable = true)
 |-- maiden: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthplace: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- healthcare_expenses: decimal(18,2) (nullable = true)
 |-- healthcare_coverage: decimal(18,2) (nullable = true)



In [19]:
# 6. VALIDATE PATIENT KEYS, DATES, AND FINANCIAL VALUES

patient_validation = (
    patients_silver
    .agg(
        F.count("*").alias("row_count"),

        F.countDistinct(
            "id"
        ).alias("distinct_patient_ids"),

        F.sum(
            F.when(
                F.col("id").isNull(),
                1
            ).otherwise(0)
        ).alias("null_patient_ids"),

        F.sum(
            F.when(
                F.col("birthdate").isNull(),
                1
            ).otherwise(0)
        ).alias("null_birthdates"),

        F.sum(
            F.when(
                F.col("deathdate") < F.col("birthdate"),
                1
            ).otherwise(0)
        ).alias("death_before_birth"),

        F.sum(
            F.when(
                F.col("healthcare_expenses") < 0,
                1
            ).otherwise(0)
        ).alias("negative_healthcare_expenses"),

        F.sum(
            F.when(
                F.col("healthcare_coverage") < 0,
                1
            ).otherwise(0)
        ).alias("negative_healthcare_coverage")
    )
)

display(patient_validation)

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e80e55e0-12e8-4118-ad15-a8421698d469)

In [20]:
# 7. PROFILE PATIENT CATEGORICAL VALUES

categorical_columns = [
    "marital",
    "race",
    "ethnicity",
    "gender",
    "state"
]

for column_name in categorical_columns:

    print(
        f"Distinct values for {column_name}"
    )

    display(
        patients_silver
        .groupBy(column_name)
        .count()
        .orderBy(
            F.desc("count")
        )
    )

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 22, Finished, Available, Finished, False)

Distinct values for marital


SynapseWidget(Synapse.DataFrame, 9747d11a-6399-477f-8873-be69c184cb8b)

Distinct values for race


SynapseWidget(Synapse.DataFrame, dbf5e041-6f6a-4cf3-bbe7-287f0ce01565)

Distinct values for ethnicity


SynapseWidget(Synapse.DataFrame, 3217daf5-68e8-494c-84d7-0c32dff10f51)

Distinct values for gender


SynapseWidget(Synapse.DataFrame, 0ef0dfd4-b3a1-4759-91a0-670b994c8835)

Distinct values for state


SynapseWidget(Synapse.DataFrame, c0e922cf-481c-445b-84b3-21f4eec685aa)

In [21]:
# 8. CHECK FOR EXACT DUPLICATE PATIENT RECORDS
patient_duplicate_count = (
    patients_silver.count()
    -
    patients_silver.dropDuplicates().count()
)

print(
    f"Exact duplicate patient rows: "
    f"{patient_duplicate_count:,}"
)

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 23, Finished, Available, Finished, False)

Exact duplicate patient rows: 0


In [22]:
# ============================================================
# 9. ADD PATIENT RECORD HASH AND PROCESSING METADATA
# ============================================================

# Use the transformed business columns to build the hash.
# At this point, metadata columns have not yet been added.
patient_hash_columns = patients_silver.columns

patient_hash_inputs = [
    F.coalesce(
        F.col(column_name).cast("string"),
        F.lit("<NULL>")
    )
    for column_name in patient_hash_columns
]

patients_silver = (
    patients_silver

    # Create a deterministic SHA-256 business-record hash
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *patient_hash_inputs
            ),
            256
        )
    )

    # Add processing and lineage metadata
    .withColumn(
        "source_system",
        F.lit(SOURCE_SYSTEM)
    )
    .withColumn(
        "source_table",
        F.lit(SOURCE_TABLE)
    )
    .withColumn(
        "source_file",
        F.lit(SOURCE_FILE)
    )
    .withColumn(
        "silver_processed_timestamp",
        F.current_timestamp()
    )
)

print(
    f"Patient hash input columns: "
    f"{len(patient_hash_columns)}"
)

print(
    "Patient record hash and Silver processing "
    "metadata were added."
)

patients_silver.select(
    "id",
    "record_hash",
    "source_system",
    "source_file",
    "silver_processed_timestamp"
).show(
    5,
    truncate=False
)

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 24, Finished, Available, Finished, False)

Patient hash input columns: 25
Patient record hash and Silver processing metadata were added.
+------------------------------------+----------------------------------------------------------------+-------------------------+------------+--------------------------+
|id                                  |record_hash                                                     |source_system            |source_file |silver_processed_timestamp|
+------------------------------------+----------------------------------------------------------------+-------------------------+------------+--------------------------+
|b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85|b3890aac4f3b9d30f4312bbb1d1176120f85435f9ed3d7f122632bb095d4735b|Synthea CSV November 2021|patients.csv|2026-10-22 02:40:46.855715|
|c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8|1e74886ed23822957323db22115b5d47608202c81bced0e7d261b5246f2131f4|Synthea CSV November 2021|patients.csv|2026-10-22 02:40:46.855715|
|339144f8-50e1-633e-a013-f361391c4cff|0826a10e68a6a75b83

In [23]:
# 10. WRITE THE PATIENTS TABLE TO THE SILVER LAYER

patient_target_table = (
    f"{SILVER_SCHEMA}.patients"
)

(
    patients_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(patient_target_table)
)

print(
    f"Silver table written successfully: "
    f"{patient_target_table}"
)

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 25, Finished, Available, Finished, False)

Silver table written successfully: silver.patients


In [24]:
# ============================================================
# 11. VERIFY THE SAVED SILVER PATIENTS TABLE
# ============================================================

saved_silver_patients = spark.table(
    f"{SILVER_SCHEMA}.patients"
)

saved_patient_validation = (
    saved_silver_patients
    .agg(
        F.count("*").alias(
            "saved_row_count"
        ),

        F.countDistinct(
            "id"
        ).alias(
            "saved_distinct_patient_ids"
        ),

        F.sum(
            F.when(
                F.col("record_hash").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_record_hashes"
        ),

        F.sum(
            F.when(
                F.length("record_hash") != 64,
                1
            ).otherwise(0)
        ).alias(
            "invalid_hash_lengths"
        ),

        F.sum(
            F.when(
                F.col("source_system").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_source_systems"
        ),

        F.sum(
            F.when(
                F.col("source_file").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_source_files"
        ),

        F.sum(
            F.when(
                F.col(
                    "silver_processed_timestamp"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_processed_timestamps"
        )
    )
)

display(
    saved_patient_validation
)

saved_patient_result = (
    saved_patient_validation.first()
)

assert saved_patient_result[
    "saved_row_count"
] == 1163, (
    "Patient row-count reconciliation failed."
)

assert saved_patient_result[
    "saved_distinct_patient_ids"
] == 1163, (
    "Patient ID uniqueness validation failed."
)

assert saved_patient_result[
    "null_record_hashes"
] == 0, (
    "Null patient record hashes were detected."
)

assert saved_patient_result[
    "invalid_hash_lengths"
] == 0, (
    "Invalid patient SHA-256 hash lengths were detected."
)

assert saved_patient_result[
    "null_source_systems"
] == 0

assert saved_patient_result[
    "null_source_files"
] == 0

assert saved_patient_result[
    "null_processed_timestamps"
] == 0

print(
    "Silver patient row-count, uniqueness, hash "
    "and metadata validations passed."
)

saved_silver_patients.printSchema()

display(
    saved_silver_patients.limit(10)
)

StatementMeta(, d2458246-d7e8-4897-854a-2f53b4b7a1e0, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f6fa0d0f-4e0c-4c27-9e62-0be414f18ecf)

Silver patient row-count, uniqueness, hash and metadata validations passed.
root
 |-- id: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- deathdate: date (nullable = true)
 |-- ssn: string (nullable = true)
 |-- drivers: string (nullable = true)
 |-- passport: string (nullable = true)
 |-- prefix: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- suffix: string (nullable = true)
 |-- maiden: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthplace: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- county: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- healthcare_expenses: decimal(18,2) (nullable = tru

SynapseWidget(Synapse.DataFrame, 1b593b3e-7c88-4b8b-855e-3c17d1b36830)